In [21]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

env = 'local'
# env = 'colab'

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# 载入final_oos_df
if env == 'local':
    final_oos_df = pd.read_parquet('/Users/jianbinchen/Downloads/results/fused_k1_forecasts_20260803_0631.parquet') 
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'
elif env == 'colab':
    import subprocess
    import google
    google.colab.drive.mount("/content/drive")
    path = '/content/drive/MyDrive/Colab Notebooks/'

    def read_drive_parquet(link):
        file_id = link.split("/d/")[1].split("/")[0]
        path = "/content/final_oos_df.parquet"
        url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
        subprocess.run(["wget", "-q", "-O", path, url], check=True)
        return pd.read_parquet(path)

    final_oos_df = read_drive_parquet(
        "https://drive.google.com/file/d/1YxdDkmTlPxexo6uD-q5b2_YqsNA8tzLb/view?usp=drive_link")

display(pd.concat([final_oos_df.head(5), final_oos_df.tail(5)], ignore_index=True))

,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k1,[1.0],[0.0],[0.025033660233020782],[0.1106194332242012],None
1,10002,2007-01-31,0.033054,fused_normal_k1,[1.0],[0.0],[0.036503858864307404],[0.08119947463274002],None
2,10025,2007-01-31,-0.028094,fused_normal_k1,[1.0],[0.0],[0.03198731690645218],[0.13557031750679016],None
3,10026,2007-01-31,-0.038517,fused_normal_k1,[1.0],[0.0],[0.04554497450590134],[0.09761743992567062],None
4,10032,2007-01-31,-0.023810,fused_normal_k1,[1.0],[0.0],[0.006967507302761078],[0.16134624183177948],None
5,93374,2024-12-31,0.012492,fused_normal_k1,[1.0],[0.0],[0.007465377449989319],[0.0639798641204834],None
6,93397,2024-12-31,-0.057236,fused_normal_k1,[1.0],[0.0],[0.00317537784576416],[0.09920903295278549],None
7,93426,2024-12-31,-0.007243,fused_normal_k1,[1.0],[0.0],[-0.004496246576309204],[0.10293024778366089],None
8,93434,2024-12-31,0.147685,fused_normal_k1,[1.0],[0.0],[-0.02898385003209114],[0.21092328429222107],None
9,93436,2024-12-31,0.001882,fused_normal_k1,[1.0],[0.0],[-0.04214758425951004],[0.1502065360546112],None


In [28]:
# =================================================================
# 1. 批量计算 Mixture VaR 和 ES
# =================================================================

def mixture_cdf_batch(x, pi, mu, sigma, nu=None):
    """
    批量计算 mixture CDF。

    x:     (N, A)，N 是股票数量，A 是 alpha 数量
    pi等:  (N, K)，K 是 mixture component 数量
    """
    z = (x[..., None] - mu[:, None, :]) / sigma[:, None, :]

    if nu is None:
        component_cdf = norm.cdf(z)
    else:
        component_cdf = t.cdf(z, df=nu[:, None, :])

    return np.sum(pi[:, None, :] * component_cdf, axis=2)


def calculate_var_es_batch(alphas, pi, mu, sigma, nu=None, iterations=45):
    """
    使用批量二分法计算 VaR，然后使用解析公式计算 ES。

    左尾 alpha <= 0.5：
        VaR 是收益率分位数；
        ES 是正的损失幅度，与原代码定义一致。

    右尾 alpha > 0.5：
        ES 是右尾平均收益，与原代码定义一致。
    """
    alphas = np.asarray(alphas, dtype=float)

    # 构造足够宽的 VaR 搜索区间
    tail_probability = min(float(alphas.min()),float(1.0 - alphas.max()))
    bracket_probability = max(tail_probability * 0.1, 1e-10)

    if nu is None:
        lower_component = (mu + sigma * norm.ppf(bracket_probability))
        upper_component = (mu + sigma * norm.ppf(1.0 - bracket_probability))
    else:
        lower_component = (mu + sigma * t.ppf(bracket_probability, df=nu))
        upper_component = (mu + sigma * t.ppf(1.0 - bracket_probability, df=nu))

    lower = np.repeat(np.min(lower_component, axis=1)[:, None],len(alphas),axis=1)
    upper = np.repeat(np.max(upper_component, axis=1)[:, None],len(alphas),axis=1)

    # 批量二分法求 VaR
    for _ in range(iterations):
        middle = 0.5 * (lower + upper)

        cdf_middle = mixture_cdf_batch(
            middle, pi, mu, sigma, nu
        )

        lower = np.where(
            cdf_middle < alphas[None, :],
            middle,
            lower
        )
        upper = np.where(
            cdf_middle >= alphas[None, :],
            middle,
            upper
        )

    var_matrix = 0.5 * (lower + upper)

    # 根据 VaR 计算 ES
    d = (
        var_matrix[:, :, None] - mu[:, None, :]
    ) / sigma[:, None, :]

    if nu is None:
        component_cdf = norm.cdf(d)
        adjusted_pdf = norm.pdf(d)
    else:
        nu_3d = nu[:, None, :]
        component_cdf = t.cdf(d, df=nu_3d)
        adjusted_pdf = (
            (nu_3d + d**2) / (nu_3d - 1.0)
        ) * t.pdf(d, df=nu_3d)

    # 左尾 partial moment
    left_partial_moment = (
        mu[:, None, :] * component_cdf
        - sigma[:, None, :] * adjusted_pdf
    )

    left_es = -np.sum(
        pi[:, None, :] * left_partial_moment,
        axis=2
    ) / alphas[None, :]

    # 右尾 partial moment
    right_partial_moment = (
        mu[:, None, :] * (1.0 - component_cdf)
        + sigma[:, None, :] * adjusted_pdf
    )

    right_es = np.sum(
        pi[:, None, :] * right_partial_moment,
        axis=2
    ) / (1.0 - alphas[None, :])

    es_matrix = np.where(
        alphas[None, :] > 0.5,
        right_es,
        left_es
    )

    return var_matrix, es_matrix


def decode_matrix(series):
    """
    将 Parquet 中保存的 JSON 字符串或 list 转成二维数组。
    """
    first_value = series.iloc[0]

    if isinstance(first_value, str):
        values = series.map(json.loads).to_numpy()
    else:
        values = series.to_numpy()

    return np.stack(values).astype(float, copy=False)

In [29]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_test = "full"  # "quick" or "full"

# 需要计算的 alpha
alphas = np.array([0.01,0.05, 0.10], dtype=float)
# 如果要同时计算多个 alpha，只改这一行：
# alphas = np.array([0.01, 0.05, 0.10], dtype=float)

# 每次处理多少行；内存不足可以改成 20_000
chunk_size = 50_000

# quick 模式只计算前 100 行
if run_test == "quick":
    calculation_df = final_oos_df.iloc[:100].copy()
else:
    calculation_df = final_oos_df

total = len(calculation_df)

# =================================================================
# 1. 一次性解码 mixture 参数
# =================================================================
pi = decode_matrix(calculation_df["pi_vec"])
mu = decode_matrix(calculation_df["mu_vec"])
sigma = decode_matrix(calculation_df["sigma_vec"])

# 判断是 Normal mixture 还是 Student-t mixture
nu = None
if (
    "nu_vec" in calculation_df.columns
    and calculation_df["nu_vec"].notna().any()
):
    nu = decode_matrix(calculation_df["nu_vec"])

y_true = calculation_df["target_ret_final"].to_numpy(dtype=float)

# 保存所有 alpha 的计算结果
var_matrix = np.empty((total, len(alphas)), dtype=float)
es_matrix = np.empty((total, len(alphas)), dtype=float)

# =================================================================
# 2. 分块计算 VaR 和 ES
# =================================================================
for start in range(0, total, chunk_size):
    stop = min(start + chunk_size, total)
    index = slice(start, stop)

    chunk_nu = None if nu is None else nu[index]

    chunk_var, chunk_es = calculate_var_es_batch(
        alphas=alphas,
        pi=pi[index],
        mu=mu[index],
        sigma=sigma[index],
        nu=chunk_nu
    )

    var_matrix[index] = chunk_var
    es_matrix[index] = chunk_es

# =================================================================
# 3. 保留原来的预测均值和预测波动率
# =================================================================
mean_pred = np.sum(pi * mu, axis=1)

if nu is None:
    component_var = sigma**2
else:
    component_var = sigma**2 * (nu / (nu - 2.0))

mixture_variance = (
    np.sum(pi * (mu**2 + component_var), axis=1)
    - mean_pred**2
)

vol_pred = np.sqrt(np.maximum(mixture_variance, 0.0))

# =================================================================
# 4. 构造最终 ES 表格
# =================================================================
result_data = {
    "permno": calculation_df["permno"].to_numpy(),
    "mthcaldt": calculation_df["mthcaldt"].to_numpy(),
}

for column, alpha in enumerate(alphas):
    alpha_label = f"{alpha * 100:g}".replace(".", "_")

    result_data[f"es_{alpha_label}"] = es_matrix[:, column]
    result_data[f"var_{alpha_label}"] = var_matrix[:, column]

# 为了不影响后面的资产定价检验，继续保留这三列
result_data["mean_pred"] = mean_pred
result_data["vol_pred"] = vol_pred
result_data["realized_ret"] = y_true

ES_df = pd.DataFrame(result_data)

display(pd.concat([ES_df.head(4), ES_df.tail(4)]))

,permno,mthcaldt,es_1,var_1,es_5,var_5,es_10,var_10,mean_pred,vol_pred,realized_ret
0,10001,2007-01-31,0.269791,-0.232306,0.203142,-0.156919,0.169102,-0.116731,0.025034,0.110619,0.264774
1,10002,2007-01-31,0.179910,-0.152394,0.130987,-0.097057,0.106000,-0.067557,0.036504,0.081199,0.033054
2,10025,2007-01-31,0.329337,-0.283396,0.247655,-0.191006,0.205936,-0.141753,0.031987,0.135570,-0.028094
3,10026,2007-01-31,0.214626,-0.181547,0.155812,-0.115021,0.125772,-0.079557,0.045545,0.097617,-0.038517
683375,93397,2024-12-31,0.261238,-0.227619,0.201464,-0.160009,0.170935,-0.123966,0.003175,0.099209,-0.057236
683376,93426,2024-12-31,0.278827,-0.243948,0.216812,-0.173801,0.185137,-0.136407,-0.004496,0.102930,-0.007243
683377,93434,2024-12-31,0.591140,-0.519665,0.464058,-0.375922,0.399151,-0.299293,-0.028984,0.210923,0.147685
683378,93436,2024-12-31,0.442480,-0.391580,0.351981,-0.289215,0.305758,-0.234645,-0.042148,0.150207,0.001882


In [30]:
ES_df.to_parquet(os.path.join(path, f"ES_df_{timestamp}.parquet"),index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output


### ---------------------------
### 旧代码, 准备删掉 
### ---------------------------


In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_test = "quick"  # "quick" or "full"
alphas = [0.01, 0.05]

# =================================================================
# 2. 主程序：从 SQLite 读取数据并执行计算
# =================================================================
results = []

total = len(final_oos_df)
print_every = max(1, total // 20)  # 每 5% 打印一次

print(">>> Extracting Tail Risk Factors (ES & VaR)...")
for idx, (df_idx, row) in enumerate(final_oos_df.iterrows()):
    if run_test == "quick" and idx >= 100:  # 测试模式下只跑前100行
        break
    # 每 x% 打印一次进度
    if idx % print_every == 0:
        print(f"  Progress: {idx/total*100:.0f}% ({idx}/{total})")
    # 1. 解码 JSON 字符串回 Numpy 数组
    pi = np.array(json.loads(row['pi_vec']))
    mu = np.array(json.loads(row['mu_vec']))
    sigma = np.array(json.loads(row['sigma_vec']))
    y_true = row['target_ret_final'] # 真实发生的（缩放后的）收益率

    # 🌟 2. 安全读取自由度 nu (智能判断分布类型)
    nu = None
    if 'nu_vec' in row and pd.notna(row['nu_vec']):
        nu = np.array(json.loads(row['nu_vec']))

    tail_result = {}
    for alpha in alphas:
        tail_result.update(
            calculate_tail_result(alpha, pi, mu, sigma, nu)
        )
    pit_result = calculate_pit_result(y_true, pi, mu, sigma, nu)
    crps_result = calculate_crps_result(y_true, pi, mu, sigma, nu)

    # 🌟 3. 高阶矩计算 (方差会随 t 分布自由度变化)
    mean_pred = np.sum(pi * mu)
    if nu is None:
        component_var = sigma**2
    else:
        # t分布的真实方差公式: sigma^2 * (nu / (nu - 2))
        # (我们的网络中 nu_head 加了 2.01，所以 nu 必然大于 2，不会出现除以0的错误)
        component_var = sigma**2 * (nu / (nu - 2))
    # 混合分布方差: sum(pi * (mu^2 + sigma^2)) - mean^2
    var_pred = np.sum(pi * (mu**2 + component_var)) - mean_pred**2

    # 将 VaR/ES、PIT 和 CRPS 合并进当前结果
    results.append({
        'permno': row['permno'],
        'mthcaldt': row['mthcaldt'],
        **tail_result,
        **pit_result,
        **crps_result,
        'mean_pred': mean_pred,
        'vol_pred': np.sqrt(var_pred),
        'realized_ret': y_true
    })

ES_df = pd.DataFrame(results)

>>> Extracting Tail Risk Factors (ES & VaR)...
  Progress: 0% (0/683379)


In [ ]:
ES_df.to_parquet(os.path.join(path, f'ES_df_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /content/drive/MyDrive/Colab Notebooks/


In [11]:
ES_df.shape

(100, 11)

In [10]:
ES_df.head(5)

,permno,mthcaldt,es_1,var_1,es_5,var_5,pit,crps,mean_pred,vol_pred,realized_ret
0,10001,2007-01-31,0.395917,-0.248923,0.224684,-0.149783,0.976717,0.211591,-0.002818,0.120762,0.264774
1,10002,2007-01-31,0.245836,-0.183986,0.166888,-0.125492,0.668700,0.027469,0.005658,0.094750,0.033054
2,10025,2007-01-31,0.397336,-0.316369,0.285774,-0.222265,0.452710,0.031514,-0.003169,0.150648,-0.028094
3,10026,2007-01-31,0.278963,-0.199669,0.181039,-0.132394,0.328968,0.025954,0.004606,0.099829,-0.038517
4,10032,2007-01-31,0.464055,-0.376132,0.339964,-0.267740,0.499203,0.036452,-0.013386,0.173100,-0.023810


In [19]:
# if env == 'colab':
#     # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
#     google.colab.runtime.unassign()

In [ ]:
alphas = [0.01, 0.05]
list_A = [np.geomspace(1e-4, alpha, 14) for alpha in alphas]
list_A

[array([0.0001    , 0.00014251, 0.00020309, 0.00028943, 0.00041246,
        0.0005878 , 0.00083768, 0.00119378, 0.00170125, 0.00242446,
        0.00345511, 0.00492388, 0.00701704, 0.01      ]),
 array([0.0001    , 0.00016129, 0.00026015, 0.0004196 , 0.00067679,
        0.0010916 , 0.00176067, 0.00283982, 0.00458041, 0.00738784,
        0.01191601, 0.01921958, 0.03099966, 0.05      ])]

In [20]:
[np.geomspace(1e-4, alpha, 14) for alpha in alphas]+[np.asarray(alphas)]

[array([0.0001    , 0.00014251, 0.00020309, 0.00028943, 0.00041246,
        0.0005878 , 0.00083768, 0.00119378, 0.00170125, 0.00242446,
        0.00345511, 0.00492388, 0.00701704, 0.01      ]),
 array([0.0001    , 0.00016129, 0.00026015, 0.0004196 , 0.00067679,
        0.0010916 , 0.00176067, 0.00283982, 0.00458041, 0.00738784,
        0.01191601, 0.01921958, 0.03099966, 0.05      ]),
 array([0.01, 0.05])]